대이터 블러오기

In [1]:
# nbformat은 requirements.txt에 포함되어 있음 (pip install -r requirements.txt)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_parquet("../eda/data_preparation/payment_type_dataset.parquet")

# 원본(80MB)은 저장소에 포함하지 않는다. README의 URL에서 받으면 아래 비교 셀도 실행된다.
# 없으면 원본 비교만 건너뛰고 이후 분석은 전처리본(df)으로 그대로 진행한다.
RAW_PATH = Path("../eda/yellow_tripdata_2026-05_snappy.parquet")
raw_df = pd.read_parquet(RAW_PATH) if RAW_PATH.exists() else None

In [3]:
df.head()

,trip_distance,fare_amount,trip_duration,cbd_congestion_fee,tolls_amount,PULocationID,DOLocationID,hour,day_of_week,VendorID,is_airport,payment_label
0,7.51,35.9,27.816667,0.00,0.0,138,37,0,4,2,1,현금
1,6.14,27.5,22.700000,0.75,0.0,138,237,0,4,2,1,신용카드
2,2.40,19.1,19.950000,0.75,0.0,249,232,0,4,1,0,신용카드
3,1.20,9.3,7.600000,0.75,0.0,232,114,0,4,1,0,신용카드
4,5.89,28.9,25.333333,0.00,0.0,255,236,0,4,2,0,신용카드


In [4]:
# 원본이 있을 때만 표시된다 (아래 출력은 원본을 두고 실행했을 때의 결과)
raw_df.head() if raw_df is not None else "원본 파일 없음 — 건너뜀"

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1.0,7.51,1.0,N,138,37,2,35.9,6.00,0.5,0.00,0.0,1.0,45.40,0.0,2.0,0.00
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1.0,6.14,1.0,N,138,237,1,27.5,6.00,0.5,9.56,0.0,1.0,49.81,2.5,2.0,0.75
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1.0,2.40,1.0,N,249,232,1,19.1,4.25,0.5,3.73,0.0,1.0,28.58,2.5,0.0,0.75
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,0.0,1.20,1.0,N,232,114,1,9.3,4.25,0.5,3.00,0.0,1.0,18.05,2.5,0.0,0.75
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2.0,0.86,1.0,N,140,237,1,7.2,0.00,0.5,2.44,0.0,1.0,14.64,2.5,0.0,0.00


In [5]:
# 원본 데이터와 비교
import numpy as np
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("df :", df.shape)
if raw_df is not None:
    print("raw :", raw_df.shape, f"(유지율 {len(df)/len(raw_df):.2%})")
else:
    print("raw : 원본 파일 없음 — 유지율 95.12% (전처리 단계 기록 기준)")
print()
df.info()

df : (3891255, 12)
raw : (4090836, 20) (유지율 95.12%)

<class 'pandas.DataFrame'>
RangeIndex: 3891255 entries, 0 to 3891254
Data columns (total 12 columns):
 #   Column              Dtype  
---  ------              -----  
 0   trip_distance       float64
 1   fare_amount         float64
 2   trip_duration       float64
 3   cbd_congestion_fee  float64
 4   tolls_amount        float64
 5   PULocationID        int32  
 6   DOLocationID        int32  
 7   hour                int32  
 8   day_of_week         int32  
 9   VendorID            int32  
 10  is_airport          int64  
 11  payment_label       str    
dtypes: float64(5), int32(5), int64(1), str(1)
memory usage: 322.0 MB


유지율 95%라서 원본 그냥 버리고 은혜님께서 진행한 데이터 믿고 변환하긔

In [6]:
print(df["payment_label"].value_counts())
print()
print(df["payment_label"].value_counts(normalize=True).round(4))

payment_label
신용카드         2660128
Flex Fare     878188
현금            352939
Name: count, dtype: int64

payment_label
신용카드         0.6836
Flex Fare    0.2257
현금           0.0907
Name: proportion, dtype: float64


In [7]:
# 컬럼별 dtype·결측·고유값 요약
summary = pd.DataFrame({
    "dtype" : df.dtypes.astype(str),
    "결측 수" : df.isna().sum(),
    "결측%" : (df.isna().mean() * 100).round(2),
    "고유값" : df.nunique(),
})
summary

,dtype,결측 수,결측%,고유값
trip_distance,float64,0,0.0,4740
fare_amount,float64,0,0.0,11401
trip_duration,float64,0,0.0,9264
cbd_congestion_fee,float64,0,0.0,2
tolls_amount,float64,0,0.0,1082
PULocationID,int32,0,0.0,259
DOLocationID,int32,0,0.0,260
hour,int32,0,0.0,24
day_of_week,int32,0,0.0,7
VendorID,int32,0,0.0,3


In [8]:
NUM_COLS = ["trip_distance", "fare_amount", "trip_duration",
            "cbd_congestion_fee", "tolls_amount"]
CAT_COLS = ["PULocationID", "DOLocationID", "hour",
            "day_of_week", "VendorID", "is_airport"]
TARGET   = "payment_label"

# 기술통계 — 평균·표준편차·분위수(25/50/75%) 포함
df[NUM_COLS].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
trip_distance,3891255.0,3.520,4.269,0.010,1.1,1.940,3.95,199.300
fare_amount,3891255.0,21.462,17.326,0.010,10.0,16.300,26.50,1061.720
trip_duration,3891255.0,18.816,15.417,0.017,8.9,14.683,23.45,179.983
cbd_congestion_fee,3891255.0,0.501,0.353,0.000,0.0,0.750,0.75,0.750
tolls_amount,3891255.0,0.548,2.183,0.000,0.0,0.000,0.00,145.600


In [9]:
# 왜도 확인 — log1p 변환 전후 비교
pd.DataFrame({
    "skew":      df[NUM_COLS].skew().round(2),
    "skew_log1p": np.log1p(df[NUM_COLS].clip(lower=0)).skew().round(2),
})

,skew,skew_log1p
trip_distance,3.32,0.90
fare_amount,3.27,0.40
trip_duration,2.48,-0.01
cbd_congestion_fee,-0.71,-0.71
tolls_amount,5.34,3.51


플롯틀리 확인

In [10]:
# 3클래스 색·순서 고정. 아래 그림은 전부 groupby 집계 기반 (389만 행 raw scatter는 안 그림)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ORDER = ["신용카드", "현금", "Flex Fare"]
COLOR = {"신용카드": "#2a78d6", "현금": "#eb6834", "Flex Fare": "#1baf7a"}
px.defaults.template = "plotly_white"

LAYOUT = dict(template="plotly_white", height=430,
              margin=dict(l=60, r=30, t=60, b=50),
              legend=dict(orientation="h", y=1.06, x=0, title=""),
              hovermode="closest")

In [11]:
# 지역ID를 정수로 넣으면 x축(ID 크기) 따라 추세가 있어야 하는데 전혀 없음 -> 타겟 인코딩으로 가야 함
zone = (df.groupby("PULocationID")["payment_label"]
          .value_counts(normalize=True).unstack(fill_value=0)
          .reindex(columns=ORDER, fill_value=0)
          .join(df["PULocationID"].value_counts().rename("n"))
          .query("n >= 500").reset_index())

fig = go.Figure()
for c in ORDER:
    fig.add_scatter(
        x=zone["PULocationID"], y=zone[c], mode="markers", name=c,
        marker=dict(color=COLOR[c], sizemode="area", size=zone["n"],
                    sizeref=zone["n"].max() / 22**2, sizemin=4,
                    line=dict(color="white", width=2)),
        customdata=zone["n"],
        hovertemplate=f"{c}<br>PU %{{x}} · %{{y:.1%}}<br>n=%{{customdata:,}}<extra></extra>")

# 눈에 띄는 두 곳만 직접 라벨 (나머지는 hover)
for zid, txt in [(132, "132 JFK"), (138, "138 LGA")]:
    r = zone.loc[zone["PULocationID"] == zid].iloc[0]
    fig.add_annotation(x=zid, y=r["현금"], text=txt, yshift=22,
                       showarrow=False, font=dict(size=11, color="#555"))

fig.update_layout(title="승차지역 ID(정수) vs 결제수단 비율 — 순서에 아무 의미 없음",
                  xaxis_title="PULocationID (정수 그대로)", yaxis_title="비율",
                  yaxis_tickformat=".0%", **LAYOUT)
fig.show()

In [12]:
# 같은 승차지역 안에서도 목적지에 따라 비율이 갈림 (JFK는 현금 6.5%~98%) -> 지역 단독보다 경로(PU_DO) 인코딩
top_pu = df["PULocationID"].value_counts().head(12).index
route = (df[df["PULocationID"].isin(top_pu)]
           .groupby(["PULocationID", "DOLocationID"])["payment_label"]
           .agg(n="size",
                현금=lambda s: (s == "현금").mean(),
                Flex_Fare=lambda s: (s == "Flex Fare").mean())
           .query("n >= 200").reset_index())
order_pu = route.groupby("PULocationID")["현금"].std().sort_values(ascending=False).index

fig = go.Figure()
for c, col in [("현금", "현금"), ("Flex Fare", "Flex_Fare")]:
    fig.add_box(x=route["PULocationID"].astype(str), y=route[col], name=c,
                marker_color=COLOR[c], line_width=1.5, boxpoints=False)

fig.update_layout(title="승차지역별 · 경로 단위 결제수단 비율의 퍼짐 (경로 n≥200)",
                  boxmode="group", xaxis_title="PULocationID",
                  yaxis_title="해당 경로의 비율", yaxis_tickformat=".0%",
                  xaxis=dict(categoryorder="array",
                             categoryarray=[str(i) for i in order_pu]),
                  **LAYOUT)
fig.show()

In [13]:
# 카드/현금 곡선이 거의 포개짐(d≈0) — Flex만 떨어짐. 원본 수치형은 카드vs현금엔 못 씀
cols = ["trip_distance", "fare_amount", "trip_duration"]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.07)

for j, col in enumerate(cols, 1):
    edges = np.histogram_bin_edges(np.log1p(df[col].clip(lower=0)), bins=70)
    mid = (edges[:-1] + edges[1:]) / 2
    for c in ORDER:
        v = np.log1p(df.loc[df[TARGET] == c, col].clip(lower=0))
        dens, _ = np.histogram(v, bins=edges, density=True)
        fig.add_scatter(x=mid, y=dens, mode="lines", name=c, legendgroup=c,
                        showlegend=(j == 1), line=dict(color=COLOR[c], width=2),
                        row=1, col=j)
    fig.update_xaxes(title_text=f"log1p({col})", row=1, col=j)

fig.update_layout(title="결제수단별 분포 — 카드와 현금이 겹침", **LAYOUT)
fig.update_yaxes(title_text="밀도", row=1, col=1)
fig.show()

# 숫자로도 확인: 카드-현금 중앙값 차이가 거의 없음
df.groupby(TARGET)[cols].median().reindex(ORDER).round(2)

,trip_distance,fare_amount,trip_duration
payment_label,,,
신용카드,1.74,14.2,13.88
현금,1.59,13.5,12.72
Flex Fare,2.88,23.1,17.33


In [14]:
# JFK 승차만. 현금이 거리 따라 쭉 주는 게 아니라 10마일에서 꺾임 = 정액제 구간 시작
jfk = df[df["PULocationID"] == 132]
BINS = [0, 2, 5, 10, 14, 18, 25]
comp = (jfk.groupby(pd.cut(jfk["trip_distance"], BINS), observed=True)[TARGET]
           .value_counts(normalize=True).unstack(fill_value=0)
           .reindex(columns=ORDER, fill_value=0))
labels = [f"{a}~{b}mi" for a, b in zip(BINS[:-1], BINS[1:])]

fig = go.Figure()
for c, pos in zip(ORDER, ["top center", "bottom center", "top center"]):
    fig.add_scatter(x=labels, y=comp[c], mode="lines+markers+text", name=c,
                    line=dict(color=COLOR[c], width=2), marker=dict(size=9),
                    text=[f"{v:.0%}" for v in comp[c]], textposition=pos,
                    textfont=dict(size=10, color="#555"))

fig.add_vline(x=2.5, line_dash="dot", line_color="#999",
              annotation_text="여기서 꺾임 (정액제 시작)", annotation_position="top right")
fig.update_layout(title=f"JFK 승차 거리구간별 결제수단 비율 (n={len(jfk):,})",
                  xaxis_title="trip_distance", yaxis_title="비율",
                  yaxis_tickformat=".0%", yaxis_range=[-0.08, 1.08], **LAYOUT)
fig.show()

In [15]:
# $70 정액 봉우리에 카드가 몰림. JFK 안에 요금체계가 2개 -> 전체 평균에선 상쇄돼서 d≈0이었던 것
fig = go.Figure()
edges = np.arange(0, 121, 2.0)
mid = (edges[:-1] + edges[1:]) / 2
for c in ORDER:
    cnt, _ = np.histogram(jfk.loc[jfk[TARGET] == c, "fare_amount"], bins=edges)
    fig.add_scatter(x=mid, y=cnt, mode="lines", name=c,
                    line=dict(color=COLOR[c], width=2))

fig.add_vrect(x0=68, x1=72, fillcolor="#999", opacity=0.12, line_width=0,
              annotation_text="$68~72 정액", annotation_position="top left")
fig.update_layout(title="JFK 승차 요금 분포 — 미터기 봉우리 + $70 정액 봉우리",
                  xaxis_title="fare_amount ($)",
                  yaxis_title="건수 (로그축)", yaxis_type="log", **LAYOUT)
fig.show()

flat = jfk["fare_amount"].between(68, 72)
pd.DataFrame({
    "n": [int(flat.sum()), int((~flat).sum())],
    "현금비율": [(jfk.loc[flat, TARGET] == "현금").mean().round(3),
              (jfk.loc[~flat, TARGET] == "현금").mean().round(3)],
}, index=["정액($68~72)", "그 외(미터기)"])

,n,현금비율
정액($68~72),71058,0.124
그 외(미터기),70039,0.420
